# 한국어 임베딩 모델 Fine-tuning

이 노트북은 KLUE RoBERTa 모델을 사용하여 한국어 문장 임베딩 모델을 학습합니다.
KLUE STS(Semantic Textual Similarity) 데이터셋을 활용하여 문장 간 의미적 유사도를 측정할 수 있는 모델을 만듭니다.


## 1. 임베딩 모델 초기화

**klue/roberta-base**를 기반으로 SentenceTransformer 모델을 만듭니다:
- `Transformer`: 사전학습된 RoBERTa 모델을 로드 (토큰 임베딩 생성)
- `Pooling`: 여러 토큰의 임베딩을 하나의 문장 임베딩으로 변환 (평균 풀링 사용)
- 이 두 레이어를 조합하여 최종 임베딩 모델을 생성합니다


In [2]:
from sentence_transformers import SentenceTransformer, models

# Transformer 레이어만 가져오기 (전체 SentenceTransformer가 아닌)
word_embedding_model = models.Transformer("klue/roberta-base")

# Pooling 레이어 추가
pooling_layer = models.Pooling(
  word_embedding_model.get_word_embedding_dimension(),
  pooling_mode_mean_tokens=True,
)

# 최종 embedding 모델 생성
embedding_model = SentenceTransformer(
  modules=[word_embedding_model, pooling_layer]
)

Some weights of RobertaModel were not initialized from the model checkpoint at klue/roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


## 2. 데이터셋 로드

**KLUE STS** 데이터셋을 로드합니다:
- 한국어 문장 쌍과 그들 간의 유사도 점수(0~5)를 포함
- `train`: 학습용 데이터
- `validation`: 테스트용 데이터로 사용
- 예시: 두 문장이 얼마나 의미적으로 유사한지 점수로 표현


In [5]:
from datasets import load_dataset

klue_sts_train = load_dataset("klue", "sts", split="train")
klue_sts_test = load_dataset("klue", "sts", split="validation")

klue_sts_train[0]

{'guid': 'klue-sts-v1_train_00000',
 'source': 'airbnb-rtt',
 'sentence1': '숙소 위치는 찾기 쉽고 일반적인 한국의 반지하 숙소입니다.',
 'sentence2': '숙박시설의 위치는 쉽게 찾을 수 있고 한국의 대표적인 반지하 숙박시설입니다.',
 'labels': {'label': 3.7, 'real-label': 3.714285714285714, 'binary-label': 1}}

## 3. 학습/검증 데이터 분할

원래 학습 데이터를 다시 분할합니다:
- 90%는 학습용 (`klue_sts_train`)
- 10%는 검증용 (`klue_sts_eval`) - 학습 중 성능 모니터링용
- 테스트 데이터(`klue_sts_test`)는 별도로 유지


In [6]:
klue_sts_train = klue_sts_train.train_test_split(test_size=0.1, seed=42)
klue_sts_train, klue_sts_eval = klue_sts_train["train"], klue_sts_train["test"]

## 4. 데이터 변환 함수

SentenceTransformer가 사용할 수 있는 형식으로 데이터를 변환합니다:
- `InputExample`: 두 문장과 유사도 레이블을 포함
- **중요**: 레이블을 5로 나누어 0~1 범위로 정규화 (모델 학습에 적합한 범위)


In [7]:
from sentence_transformers import InputExample

def prepare_sts_examples(dataset):
  examples = []

  for data in dataset:
    examples.append(
      InputExample(
        texts=[data["sentence1"], data["sentence2"]],
        label=data["labels"]['label'] / 5.0
      )
    )
  return examples

## 5. 데이터 준비

각 데이터셋을 InputExample 형식으로 변환합니다:
- `train_examples`: 모델 학습용
- `eval_examples`: 학습 중 검증용
- `test_examples`: 최종 평가용


In [8]:
train_examples = prepare_sts_examples(klue_sts_train)
eval_examples = prepare_sts_examples(klue_sts_eval)
test_examples = prepare_sts_examples(klue_sts_test)
  

## 6. DataLoader 생성

학습 데이터를 배치로 나누어 제공하는 DataLoader를 생성합니다:
- `batch_size=16`: 한 번에 16개의 문장 쌍을 처리
- `shuffle=True`: 매 에폭마다 데이터 순서를 섞어 학습 효과 향상


In [9]:
from torch.utils.data import DataLoader
train_dataloader = DataLoader(train_examples, batch_size=16, shuffle=True)

## 7. Evaluator 설정

모델 성능을 측정하는 평가자를 생성합니다:
- `EmbeddingSimilarityEvaluator`: 문장 임베딩 간 코사인 유사도를 계산하고 실제 레이블과 비교
- Pearson 상관계수와 Spearman 상관계수를 계산하여 성능 측정
- `eval_evaluator`: 학습 중 사용
- `test_evaluator`: 최종 평가용


In [10]:
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator

eval_evaluator = EmbeddingSimilarityEvaluator.from_input_examples(eval_examples)
test_evaluator = EmbeddingSimilarityEvaluator.from_input_examples(test_examples)

## 8. 학습 전 성능 측정

Fine-tuning 하기 전 모델의 baseline 성능을 확인합니다:
- `pearson_cosine`: 0.35 (예측 유사도와 실제 유사도의 피어슨 상관계수)
- `spearman_cosine`: 0.36 (예측 유사도와 실제 유사도의 스피어만 상관계수)
- **값이 높을수록 좋음** (최대 1.0)
- 현재는 fine-tuning 전이라 성능이 낮음


In [11]:
test_evaluator(embedding_model)

{'pearson_cosine': 0.3477069870589581, 'spearman_cosine': 0.35560473197486514}

## 다음 단계: 모델 학습

아래 코드를 추가하여 실제 fine-tuning을 진행할 수 있습니다:

```python
from sentence_transformers import losses

# CosineSimilarityLoss: 문장 쌍의 코사인 유사도가 레이블과 가까워지도록 학습
train_loss = losses.CosineSimilarityLoss(embedding_model)

# 학습 실행
embedding_model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    evaluator=eval_evaluator,
    epochs=4,
    warmup_steps=100,
    output_path='./klue-roberta-sts-finetuned',
    evaluation_steps=500,  # 500 step마다 검증
    save_best_model=True
)

# 최종 테스트
final_score = test_evaluator(embedding_model)
print(f"최종 성능: {final_score}")
```
